# Replication Analysis: Student Acceptance and Interaction Patterns with Generative AI in Accessible UI Design

**Authors:** Daniela Marques, André Luiz Satoshi Kawamoto, Valéria Farinazzo Martins  
**Publication Target:** CLEI Electronic Journal (CLEI EJ)  
**Zenodo Permanent DOI:** [10.5281/zenodo.24170706](https://zenodo.org/records/24170706)  
**GitHub Repository:** [https://github.com/alkawamo/clei-genai-study](https://github.com/alkawamo/clei-genai-study)  

---

## Overview
This Google Colab / Jupyter notebook reproduces all empirical findings, statistical tables, psychometric reliability checks, and accessibility audits presented in the journal article.

The analysis processes three de-identified CSV datasets:
1. `dataset_student_profiles_and_pretest.csv` ($N = 41$)
2. `dataset_utaut_paired_pre_post.csv` ($N = 41$)
3. `dataset_prompt_iterations_and_accessibility.csv` ($N = 26$)

Zero installation is required when executed on Google Colab.

In [ ]:
# 1. Setup Environment and Automatic Data Acquisition
import os
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# Set consistent plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# Check if running in Google Colab or local environment
DATA_DIR = "../zenodo_dataset"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "zenodo_dataset"
if not os.path.exists(DATA_DIR):
    # Fetch directly from GitHub raw if in standalone Colab
    print("Downloading datasets from GitHub repository...")
    BASE_URL = "https://raw.githubusercontent.com/alkawamo/clei-genai-study/main/zenodo_dataset/"
    os.makedirs("zenodo_dataset", exist_ok=True)
    for fname in ["dataset_student_profiles_and_pretest.csv", "dataset_utaut_paired_pre_post.csv", "dataset_prompt_iterations_and_accessibility.csv"]:
        url = BASE_URL + fname
        df_temp = pd.read_csv(url)
        df_temp.to_csv(os.path.join("zenodo_dataset", fname), index=False)
    DATA_DIR = "zenodo_dataset"

print(f"Data directory verified: {DATA_DIR}")
df_profiles = pd.read_csv(os.path.join(DATA_DIR, "dataset_student_profiles_and_pretest.csv"))
df_utaut = pd.read_csv(os.path.join(DATA_DIR, "dataset_utaut_paired_pre_post.csv"))
df_accessibility = pd.read_csv(os.path.join(DATA_DIR, "dataset_prompt_iterations_and_accessibility.csv"))

print(f"Loaded Profiles: N = {len(df_profiles)}")
print(f"Loaded UTAUT Paired: N = {len(df_utaut)}")
print(f"Loaded Accessibility Audit: N = {len(df_accessibility)}")

## Section 1: Participant Demographics and Profile
Reproduces Section 4.1 and Section 5.1 of the manuscript.

In [ ]:
# Gender, Age, and Prior UI Experience
print("=== GENDER DISTRIBUTION (N = 41) ===")
gender_counts = df_profiles['gender'].value_counts()
gender_pct = df_profiles['gender'].value_counts(normalize=True) * 100
print(pd.DataFrame({'Count': gender_counts, 'Percentage (%)': gender_pct.round(1)}))

print("\n=== AGE DISTRIBUTION (N = 41) ===")
age_counts = df_profiles['age_group'].value_counts()
age_pct = df_profiles['age_group'].value_counts(normalize=True) * 100
print(pd.DataFrame({'Count': age_counts, 'Percentage (%)': age_pct.round(1)}))

print("\n=== UI DEVELOPMENT EXPERIENCE (N = 41) ===")
exp_counts = df_profiles['ui_development_experience'].value_counts()
exp_pct = df_profiles['ui_development_experience'].value_counts(normalize=True) * 100
print(pd.DataFrame({'Count': exp_counts, 'Percentage (%)': exp_pct.round(1)}))

print("\n=== PRIOR GENAI EXPERIENCE (N = 41) ===")
ai_counts = df_profiles['ai_used_before'].value_counts()
ai_pct = df_profiles['ai_used_before'].value_counts(normalize=True) * 100
print(pd.DataFrame({'Count': ai_counts, 'Percentage (%)': ai_pct.round(1)}))

## Section 2: Scale Reliability Analysis (Cronbach's Alpha)
Reproduces Table 3 of the manuscript.

In [ ]:
def cronbach_alpha(df_items):
    k = df_items.shape[1]
    item_var = df_items.var(axis=0, ddof=1).sum()
    total_var = df_items.sum(axis=1).var(ddof=1)
    if total_var == 0:
        return 0.0
    return (k / (k - 1)) * (1 - item_var / total_var)

constructs = ['PE', 'EE', 'SI', 'FC', 'BI']
alpha_results = []

for c in constructs:
    pre_items = df_utaut[[f'{c}1_pre', f'{c}2_pre', f'{c}3_pre']]
    post_items = df_utaut[[f'{c}1_post', f'{c}2_post', f'{c}3_post']]
    a_pre = cronbach_alpha(pre_items)
    a_post = cronbach_alpha(post_items)
    alpha_results.append({
        'UTAUT Construct': c,
        'Pre-Test Alpha': round(a_pre, 3),
        'Post-Test Alpha': round(a_post, 3)
    })

df_table3 = pd.DataFrame(alpha_results)
print("=== TABLE 3: CRONBACH ALPHA RELIABILITY COEFFICIENTS ===")
display(df_table3) if 'display' in globals() else print(df_table3.to_string(index=False))

## Section 3: Comparative Statistical Analysis of UTAUT Constructs
Reproduces Table 4 of the manuscript (Parametric paired t-test, Non-parametric Wilcoxon signed-rank test with zero handling, effect sizes Cohen's d).

In [ ]:
table4_results = []

for c in constructs:
    pre = df_utaut[f'{c}_pre'].values
    post = df_utaut[f'{c}_post'].values
    diff = post - pre
    
    pre_m, pre_sd = np.mean(pre), np.std(pre, ddof=1)
    post_m, post_sd = np.mean(post), np.std(post, ddof=1)
    delta_m = np.mean(diff)
    
    # Paired t-test: convention post - pre (positive t when post > pre)
    t_stat, p_t = stats.ttest_rel(post, pre)
    d_sd = np.std(diff, ddof=1)
    cohen_d = delta_m / d_sd if d_sd > 0 else 0.0
    
    # Wilcoxon signed-rank test (zero_method='wilcox' discards zero differences)
    non_zeros = diff[diff != 0]
    n_eff = len(non_zeros)
    if n_eff > 0:
        w_stat, p_w = stats.wilcoxon(post, pre, zero_method='wilcox')
    else:
        w_stat, p_w = 0.0, 1.0
        
    table4_results.append({
        'Construct': c,
        'Pre Mean (SD)': f"{pre_m:.2f} ({pre_sd:.2f})",
        'Post Mean (SD)': f"{post_m:.2f} ({post_sd:.2f})",
        'Mean Delta': f"{delta_m:+.2f}",
        't-statistic': round(t_stat, 2),
        'p-value (t)': round(p_t, 3),
        'Wilcoxon W': round(w_stat, 1),
        'N_eff': n_eff,
        'p-value (W)': round(p_w, 3),
        "Cohen's d": round(cohen_d, 2)
    })

df_table4 = pd.DataFrame(table4_results)
print("=== TABLE 4: CONSTRUCT COMPARATIVE ANALYSIS (N = 41) ===")
display(df_table4) if 'display' in globals() else print(df_table4.to_string(index=False))

## Section 4: Granular Item-by-Item Statistical Analysis
Reproduces Table 5 of the manuscript.

In [ ]:
item_summaries = {
    'PE1': 'Improved interface quality',
    'PE2': 'Increased design productivity',
    'PE3': 'Fostered design innovation',
    'EE1': 'Ease of use and lack of complication',
    'EE2': 'Rapid learning curve',
    'EE3': 'Intuitive design interaction',
    'SI1': 'Peer encouragement',
    'SI2': 'Professional community prestige',
    'SI3': 'Organizational/peer support',
    'FC1': 'Technical infrastructure availability',
    'FC2': 'Necessary resources and guidance',
    'FC3': 'Compatibility with workflows',
    'BI1': 'Intention to use in future projects',
    'BI2': 'Regular incorporation into workflows',
    'BI3': 'Willingness to recommend to peers'
}

table5_results = []
for it, summary in item_summaries.items():
    pre = df_utaut[f'{it}_pre'].values
    post = df_utaut[f'{it}_post'].values
    diff = post - pre
    non_zeros = diff[diff != 0]
    n_eff = len(non_zeros)
    if n_eff > 0:
        w_stat, p_w = stats.wilcoxon(post, pre, zero_method='wilcox')
    else:
        w_stat, p_w = 0.0, 1.0
        
    table5_results.append({
        'Item Code': it,
        'Pre Mean (SD)': f"{np.mean(pre):.2f} ({np.std(pre, ddof=1):.2f})",
        'Post Mean (SD)': f"{np.mean(post):.2f} ({np.std(post, ddof=1):.2f})",
        'Item Delta': f"{np.mean(diff):+.2f}",
        'Wilcoxon W': round(w_stat, 1),
        'N_eff': n_eff,
        'p-value': round(p_w, 3),
        'Item Summary': summary
    })

df_table5 = pd.DataFrame(table5_results)
print("=== TABLE 5: GRANULAR ITEM-BY-ITEM STATISTICAL ANALYSIS ===")
display(df_table5) if 'display' in globals() else print(df_table5.to_string(index=False))

## Section 5: Subgroup Inferential Analysis (Novices vs. Experienced)
Reproduces Table 6 of the manuscript.

In [ ]:
novices = df_utaut[df_utaut['ai_used_before'] == 'Não']
experienced = df_utaut[df_utaut['ai_used_before'] == 'Sim']

print(f"Novices: n = {len(novices)}, Experienced: n = {len(experienced)}")

table6_results = []
for c in constructs:
    d_nov = novices[f'{c}_delta'].values
    d_exp = experienced[f'{c}_delta'].values
    
    u_stat, p_u = stats.mannwhitneyu(d_nov, d_exp, alternative='two-sided')
    table6_results.append({
        'Construct': c,
        'Novices Delta (n=5)': f"{np.mean(d_nov):+.2f}",
        'Experienced Delta (n=36)': f"{np.mean(d_exp):+.2f}",
        'Mann-Whitney U': round(u_stat, 1),
        'p-value': round(p_u, 3)
    })

df_table6 = pd.DataFrame(table6_results)
print("=== TABLE 6: COMPARISON BY PRIOR GENAI EXPOSURE (MANN-WHITNEY U) ===")
display(df_table6) if 'display' in globals() else print(df_table6.to_string(index=False))

# Multiple testing correction note:
p_vals = [r['p-value'] for r in table6_results]
print("\nMultiple comparison adjustment (Bonferroni alpha threshold for 5 tests): alpha = 0.010")
for r in table6_results:
    sig = "* Significant (p < 0.05)" if r['p-value'] < 0.05 else "Not significant"
    sig_bonf = "(Survives Bonferroni p < 0.01)" if r['p-value'] < 0.01 else ""
    print(f"  {r['Construct']}: p = {r['p-value']:.3f} -> {sig} {sig_bonf}")

## Section 6: Objective Accessibility Code Audit and Prompt Evolution
Reproduces Table 8 and Table 9 of the manuscript.

In [ ]:
print("=== TABLE 8: OBJECTIVE ACCESSIBILITY COMPLIANCE AUDIT (N = 26) ===")
heuristics = [
    ('Font Size Scaling / Typography', 'has_font_size_adjustment', 'Relative units (rem/em) or enlarged base fonts (> 18px).'),
    ('Pacing and Timing Control', 'has_speed_reduction', 'Slowed interval timers (> 1200ms) or configurable delays.'),
    ('Textual Color Name Labels', 'has_text_color_labels', 'Explicit text labels inside color quadrants.'),
    ('Multimodal Audio Feedback', 'has_multimodal_audio', 'Distinct audio tone frequencies paired with visual cues.'),
    ('Full Keyboard Navigation', 'has_keyboard_navigation', 'Accessible focus management and event listeners (Tab/Enter).'),
    ('WAI-ARIA Semantic Attributes', 'has_aria_accessibility_tags', 'Semantic markup: aria-label, role="status", aria-live.'),
    ('Explicit Round State Indicator', 'has_round_feedback', 'Persistent textual display of round count to offload memory.'),
    ('High-Contrast / Dark Theme Toggle', 'has_contrast_theme_toggle', 'Persistent user-toggleable high-contrast switch.')
]

table8_data = []
for label, col, standard in heuristics:
    cnt = df_accessibility[col].sum()
    pct = (cnt / len(df_accessibility)) * 100
    table8_data.append({
        'Accessibility Dimension': label,
        'Adoption (n)': cnt,
        'Adoption (%)': f"{pct:.1f}%",
        'Evaluated Implementation Standard': standard
    })

df_table8 = pd.DataFrame(table8_data)
display(df_table8) if 'display' in globals() else print(df_table8.to_string(index=False))

print("\n=== TABLE 9: EVOLUTION ACROSS PROMPT ITERATION TIERS ===")
tier_low = df_accessibility[df_accessibility['prompt_iterations_count'] <= 3]
tier_mod = df_accessibility[df_accessibility['prompt_iterations_count'] == 4]
tier_high = df_accessibility[df_accessibility['prompt_iterations_count'] >= 5]

table9_data = []
feat_cols = [h[1] for h in heuristics]
for label, col, _ in heuristics:
    c_low = f"{tier_low[col].mean()*100:.1f}% ({tier_low[col].sum()}/{len(tier_low)})"
    c_mod = f"{tier_mod[col].mean()*100:.1f}% ({tier_mod[col].sum()}/{len(tier_mod)})"
    c_high = f"{tier_high[col].mean()*100:.1f}% ({tier_high[col].sum()}/{len(tier_high)})"
    table9_data.append({
        'Accessibility Dimension': label,
        'Low (2-3) [n=9]': c_low,
        'Moderate (4) [n=11]': c_mod,
        'High (5-7) [n=6]': c_high
    })

score_low = tier_low[feat_cols].sum(axis=1)
score_mod = tier_mod[feat_cols].sum(axis=1)
score_high = tier_high[feat_cols].sum(axis=1)

table9_data.append({
    'Accessibility Dimension': 'Mean Accessibility Score (out of 8)',
    'Low (2-3) [n=9]': f"{score_low.mean():.2f} +/- {score_low.std(ddof=1):.2f}",
    'Moderate (4) [n=11]': f"{score_mod.mean():.2f} +/- {score_mod.std(ddof=1):.2f}",
    'High (5-7) [n=6]': f"{score_high.mean():.2f} +/- {score_high.std(ddof=1):.2f}"
})
table9_data.append({
    'Accessibility Dimension': 'Normalized Compliance (%)',
    'Low (2-3) [n=9]': f"{(score_low.mean()/8)*100:.1f}%",
    'Moderate (4) [n=11]': f"{(score_mod.mean()/8)*100:.1f}%",
    'High (5-7) [n=6]': f"{(score_high.mean()/8)*100:.1f}%"
})

df_table9 = pd.DataFrame(table9_data)
display(df_table9) if 'display' in globals() else print(df_table9.to_string(index=False))

## Section 7: Correlation Analysis (Effort Expectancy Paradox)
Evaluates correlations between prompt iterations, accessibility scores, and UTAUT shifts.

In [ ]:
# Merge accessibility with UTAUT
merged = pd.merge(df_accessibility, df_utaut, on='participant_id')
merged['total_accessibility_score'] = merged[feat_cols].sum(axis=1)

r_acc, p_acc = stats.pearsonr(merged['prompt_iterations_count'], merged['total_accessibility_score'])
r_ee_post, p_ee_post = stats.pearsonr(merged['prompt_iterations_count'], merged['EE_post'])
rho_ee_delta, p_ee_delta = stats.spearmanr(merged['prompt_iterations_count'], merged['EE_delta'])

print("=== CORRELATION FINDINGS ===")
print(f"Prompt Iterations vs. Objective Accessibility Score: Pearson r = {r_acc:+.3f} (p = {p_acc:.4f})")
print(f"Prompt Iterations vs. Post-Test Effort Expectancy:   Pearson r = {r_ee_post:+.3f} (p = {p_ee_post:.4f})")
print(f"Prompt Iterations vs. Effort Expectancy Delta:       Spearman rho = {rho_ee_delta:+.3f} (p = {p_ee_delta:.4f})")

# Visualizing the Effort Expectancy Paradox and Code Quality
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.regplot(data=merged, x='prompt_iterations_count', y='total_accessibility_score', ax=axes[0], color='#2b5c8f', scatter_kws={'s': 60})
axes[0].set_title(f'Prompt Iterations vs. Accessibility Compliance\n(r = {r_acc:+.2f}, p < 0.05)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Total Prompt Iterations Count')
axes[0].set_ylabel('Objective Accessibility Score (0 to 8)')

sns.regplot(data=merged, x='prompt_iterations_count', y='EE_post', ax=axes[1], color='#c2593f', scatter_kws={'s': 60})
axes[1].set_title(f'The Effort Expectancy Paradox\n(r = {r_ee_post:+.2f}, p = {p_ee_post:.3f})', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Total Prompt Iterations Count')
axes[1].set_ylabel('Post-Test Effort Expectancy (1 to 5)')

plt.tight_layout()
plt.show()